# Seeding data for Kotlin Clothing Webshop Database

This notebook can be used to load seed data into the Kotlin Clothing Webshop application's database

## 0. Importing dependencies

In this case some additional dependencies will be needed from Maven Repository, these are the following:
- dataframe: to load into the memory the content of structured csv files, and later manipulate them
- datetime: Kotlinx-Datetime for date related operations
- kotlin-dl: This library will be used to generate embedding vectors from the articles for the recommendation system functionality.
- the postgresql driver to connect to the database management system through JDBC

In [ ]:
USE {
    repositories("https://repo.maven.apache.org/maven2")
    dependencies {
        implementation("org.jetbrains.kotlinx:dataframe:0.13.1")
        implementation("org.jetbrains.kotlinx:kotlinx-datetime:0.6.0")
        implementation("org.jetbrains.kotlinx:kotlin-deeplearning-onnx:0.5.2")
        implementation("org.postgresql:postgresql:42.7.1")
    }
}

## 1. Loading input data from csv files

To use this notebook, it is required to download the used dataset. As it was stated before, this notebook inspects the data of the [H&M Personalized Fashion Recommendations dataset](https://www.kaggle.com/competitions/h-and-m-personalized-fashion-recommendations/). The dataset is stored on Kaggle, it is possible to download it from the website, or using the [Kaggle API](https://github.com/Kaggle/kaggle-api).

After the download was successful, change value of the *pathToDownloadedCsvFiles* variable in the cell below to the path of directory of the downloaded dataset on your local machine. You probably need to adjust the value of *pathToTransformedTransactionsFile*. In this example, I'm using only the first half of the original *transactions_train.csv* file, which was generated using [the EDA Notebook](hm_dataset_inspection_eda.ipynb). The path to the original should also work, but in that case probably a lot more memory will be needed to store the data.

In [ ]:
val pathToDownloadedCsvFiles = "C:\\Egyetem\\MSc\\Onallo\\HM_dataset"
val pathToTransformedTransactionsFile = "C:\\Egyetem\\MSc\\Onallo\\HM_dataset\\transformed"

In [ ]:
val dateOfUsedTransactions = "2019-01-07"
val transactionLimit = 1000

In [ ]:
var transactionsDf = DataFrame.readCSV(
    fileOrUrl = pathToTransformedTransactionsFile + "\\transactions_train1.csv",
)

var articlesDf = DataFrame.readCSV(
    fileOrUrl = pathToDownloadedCsvFiles + "\\articles.csv",
)

var customersDf = DataFrame.readCSV(
    fileOrUrl = pathToDownloadedCsvFiles + "\\customers.csv",
    charset = Charsets.US_ASCII,
)

Let's select the transactions from the inspected period!

In [ ]:
transactionsDf = transactionsDf.filter { t_dat == LocalDate.parse(dateOfUsedTransactions) }.head(transactionLimit)

transactionsDf

Then let's keep only the articles and customers that made a transaction in the inspected period!

In [ ]:
articlesDf = articlesDf.filter { article_id in transactionsDf.article_id }

articlesDf

In [ ]:
customersDf = customersDf.filter { customer_id in transactionsDf.customer_id }

customersDf

## 2. Creating recommendation embeddings for articles

In [ ]:
val articleEmbeddings = mutableMapOf<Int, List<Float>>()

val articleIdEncodings = mutableMapOf<Int, List<Float>>()

Let's generate first the encoded values for article IDs!

In [ ]:
import org.jetbrains.kotlinx.dl.onnx.inference.OnnxInferenceModel

val articleIdEncoderModel = OnnxInferenceModel.load("models/retrieval_item_id_encoder.onnx")

val encoderInputSize = articleIdEncoderModel.inputDimensions.last().toInt()
articleIdEncoderModel.reshape(encoderInputSize.toLong())
println("Model input size: $encoderInputSize")
println("Model output shape: ${articleIdEncoderModel.outputShape.toSet()}")

var index = 0
articlesDf.forEach {
    val input = FloatArray(1)
    input[0] = index.toFloat()

    articleIdEncodings[it.article_id] = articleIdEncoderModel.predictSoftly(input).toList()
}

index += 1

articleIdEncoderModel.close()

articleIdEncodings

The categorical values needs to be one-hot encoded, let's add these categorical values as enums first!

In [ ]:
enum class GarmentGroupName {
    Accessories,
    Blouses,
    Dressed,
    DressesLadies,
    DressesSkirtsGirls,
    JerseyBasic,
    JerseyFancy,
    Knitwear,
    Outdoor,
    Shirts,
    Shoes,
    Shorts,
    Skirts,
    SocksAndTights,
    SpecialOffers,
    Swimwear,
    Trousers,
    TrousersDenim,
    UnderNightwear,
    Unknown,
    WovenJerseyKnittedMixBaby;

    companion object {

        fun getFromCsvValue(csvValue: String): GarmentGroupName = when (csvValue) {
            "Accessories" -> Accessories
            "Blouses" -> Blouses
            "Dressed" -> Dressed
            "Dresses Ladies" -> DressesLadies
            "Dresses/Skirts girls" -> DressesSkirtsGirls
            "Jersey Basic" -> JerseyBasic
            "Jersey Fancy" -> JerseyFancy
            "Knitwear" -> Knitwear
            "Outdoor" -> Outdoor
            "Shirts" -> Shirts
            "Shoes" -> Shoes
            "Shorts" -> Shorts
            "Skirts" -> Skirts
            "Socks and Tights" -> SocksAndTights
            "Special Offers" -> SpecialOffers
            "Swimwear" -> Swimwear
            "Trousers" -> Trousers
            "Trousers Denim" -> TrousersDenim
            "Under-, Nightwear" -> UnderNightwear
            "Unknown" -> Unknown
            "Woven/Jersey/Knitted mix Baby" -> WovenJerseyKnittedMixBaby
            else -> throw IllegalStateException("No GarmentGroup enum value was found for $csvValue")
        }
    }
}

enum class Shade {
    Bright,
    Dark,
    DustyLight,
    Light,
    Medium,
    MediumDusty,
    Other;

    companion object {

        fun getFromCsvValue(csvValue: String): Shade = when (csvValue) {
            "Bright" -> Bright
            "Dark" -> Dark
            "Dusty Light" -> DustyLight
            "Light" -> Light
            "Medium" -> Medium
            "Medium Dusty" -> MediumDusty
            "Other" -> Other
            else -> throw IllegalStateException("No Shade enum value was found for $csvValue")
        }
    }
}

enum class GraphicalAppearance {
    AllOverPattern,
    Application3D,
    Argyle,
    Chambray,
    Check,
    ColourBlocking,
    Contrast,
    Denim,
    Dot,
    Embroidery,
    FrontPrint,
    GlitteringMetallic,
    Hologram,
    Jacquard,
    Lace,
    Melange,
    Mesh,
    Metallic,
    MixedSolidPattern,
    Neps,
    OtherPattern,
    OtherStructure,
    PlacementPrint,
    Sequin,
    Slub,
    Solid,
    Stripe,
    Transparent,
    Treatment,
    Unknown;

    companion object {

        fun getFromCsvValue(csvValue: String): GraphicalAppearance = when (csvValue) {
            "All over pattern" -> AllOverPattern
            "Application/3D" -> Application3D
            "Argyle" -> Argyle
            "Chambray" -> Chambray
            "Check" -> Check
            "Colour blocking" -> ColourBlocking
            "Contrast" -> Contrast
            "Denim" -> Denim
            "Dot" -> Dot
            "Embroidery" -> Embroidery
            "Front print" -> FrontPrint
            "Glittering/Metallic" -> GlitteringMetallic
            "Hologram" -> Hologram
            "Jacquard" -> Jacquard
            "Lace" -> Lace
            "Melange" -> Melange
            "Mesh" -> Mesh
            "Metallic" -> Metallic
            "Mixed solid/pattern" -> MixedSolidPattern
            "Neps" -> Neps
            "Other pattern" -> OtherPattern
            "Other structure" -> OtherStructure
            "Placement print" -> PlacementPrint
            "Sequin" -> Sequin
            "Slub" -> Slub
            "Solid" -> Solid
            "Stripe" -> Stripe
            "Transparent" -> Transparent
            "Treatment" -> Treatment
            "Unknown" -> Unknown
            else -> throw IllegalStateException("No GraphicalAppearance enum value was found for $csvValue")
        }
    }
}

enum class IndexName {
    BabySizes5098,
    ChildrenAccessoriesSwimwear,
    ChildrenSizes134170,
    ChildrenSizes92140,
    Divided,
    LadiesAccessories,
    Ladieswear,
    LingeriesTights,
    Menswear,
    Sport;

    companion object {

        fun getFromCsvValue(csvValue: String): IndexName = when (csvValue) {
            "Baby Sizes 50-98" -> BabySizes5098
            "Children Accessories, Swimwear" -> ChildrenAccessoriesSwimwear
            "Children Sizes 134-170" -> ChildrenSizes134170
            "Children Sizes 92-140" -> ChildrenSizes92140
            "Divided" -> Divided
            "Ladies Accessories" -> LadiesAccessories
            "Ladieswear" -> Ladieswear
            "Lingeries/Tights" -> LingeriesTights
            "Menswear" -> Menswear
            "Sport" -> Sport
            else -> throw IllegalStateException("No IndexName enum value was found for $csvValue")
        }
    }
}

enum class PerceivedColourMasterName {
    Beige,
    Black,
    Blue,
    BluishGreen,
    Brown,
    Green,
    Grey,
    Khakigreen,
    LilacPurple,
    Metal,
    Mole,
    Orange,
    Pink,
    Red,
    Turquoise,
    undefined,
    Unknown,
    White,
    Yellow,
    YellowishGreen;

    companion object {

        fun getFromCsvValue(csvValue: String): PerceivedColourMasterName = when (csvValue) {
            "Beige" -> Beige
            "Black" -> Black
            "Blue" -> Blue
            "Bluish Green" -> BluishGreen
            "Brown" -> Brown
            "Green" -> Green
            "Grey" -> Grey
            "Khaki green" -> Khakigreen
            "Lilac Purple" -> LilacPurple
            "Metal" -> Metal
            "Mole" -> Mole
            "Orange" -> Orange
            "Pink" -> Pink
            "Red" -> Red
            "Turquoise" -> Turquoise
            "undefined" -> undefined
            "Unknown" -> Unknown
            "White" -> White
            "Yellow" -> Yellow
            "Yellowish Green" -> YellowishGreen
            else -> throw IllegalStateException("No PerceivedColourMasterName enum value was found for $csvValue")
        }
    }

}

enum class ProductGroupName {
    Accessories,
    Bags,
    Cosmetic,
    Fun,
    Furniture,
    GarmentFullBody,
    GarmentLowerBody,
    GarmentUpperBody,
    GarmentAndShoeCare,
    InteriorTextile,
    Items,
    Nightwear,
    Shoes,
    SocksAndTights,
    Stationery,
    Swimwear,
    Underwear,
    UnderwearNightwear,
    Unknown;

    companion object {

        fun getFromCsvValue(csvValue: String): ProductGroupName = when (csvValue) {
            "Accessories" -> Accessories
            "Bags" -> Bags
            "Cosmetic" -> Cosmetic
            "Fun" -> Fun
            "Furniture" -> Furniture
            "Garment Full body" -> GarmentFullBody
            "Garment Lower body" -> GarmentLowerBody
            "Garment Upper body" -> GarmentUpperBody
            "Garment and Shoe care" -> GarmentAndShoeCare
            "Interior textile" -> InteriorTextile
            "Items" -> Items
            "Nightwear" -> Nightwear
            "Shoes" -> Shoes
            "Socks & Tights" -> SocksAndTights
            "Stationery" -> Stationery
            "Swimwear" -> Swimwear
            "Underwear" -> Underwear
            "Underwear/nightwear" -> UnderwearNightwear
            "Unknown" -> Unknown
            else -> throw IllegalStateException("No ProductGroupName enum value was found for $csvValue")
        }
    }

}

The function below can be used to add a one-hot encoded value to a FloatArray!

In [ ]:
fun FloatArray.addOneHotEncoding(arrayStartIndex: Int, index: Int, total: Int) {
    for (i in 0..<total) {
        this[arrayStartIndex + i] = if (index == i) 1f else 0f
    }
}

After the inputs are properly encoded, let's generate the embedding vectors for the articles!

In [ ]:
import org.jetbrains.kotlinx.dl.onnx.inference.OnnxInferenceModel

val articleTowerModel = OnnxInferenceModel.load("models/retrieval_item_tower_model.onnx")

val towerInputSize = articleTowerModel.inputDimensions.last().toInt()
articleTowerModel.reshape(towerInputSize.toLong())
println("Model input size: $towerInputSize")
println("Model output shape: ${articleTowerModel.outputShape.toSet()}")

articlesDf.forEach {
    val input = FloatArray(towerInputSize)
    
    articleIdEncodings[it.article_id]!!.forEachIndexed { index, value -> input[index] = value }
    
    val idEncodingLength = articleIdEncodings.values.first().size
    val indexNameEncodingLength = IndexName.entries.size
    val garmentGroupEncodingLength = GarmentGroupName.entries.size
    val productGroupEncodingLength = ProductGroupName.entries.size
    val colourMasterEncodingLength = PerceivedColourMasterName.entries.size
    val graphicalAppearanceEncodingLength = GraphicalAppearance.entries.size
    
    input.addOneHotEncoding(
        arrayStartIndex = idEncodingLength, 
        index = IndexName.getFromCsvValue(index_name).ordinal, 
        total = indexNameEncodingLength,
    )
    input.addOneHotEncoding(
        arrayStartIndex = idEncodingLength + indexNameEncodingLength, 
        index = GarmentGroupName.getFromCsvValue(garment_group_name.toString()).ordinal, 
        total = garmentGroupEncodingLength,
    )
    input.addOneHotEncoding(
        arrayStartIndex = idEncodingLength + indexNameEncodingLength + garmentGroupEncodingLength, 
        index = ProductGroupName.getFromCsvValue(product_group_name.toString()).ordinal, 
        total = productGroupEncodingLength,
    )
    input.addOneHotEncoding(
        arrayStartIndex = idEncodingLength + indexNameEncodingLength + garmentGroupEncodingLength + productGroupEncodingLength, 
        index = PerceivedColourMasterName.getFromCsvValue(perceived_colour_master_name.toString()).ordinal, 
        total = colourMasterEncodingLength,
    )
    input.addOneHotEncoding(
        arrayStartIndex = idEncodingLength + indexNameEncodingLength + garmentGroupEncodingLength + productGroupEncodingLength + colourMasterEncodingLength, 
        index = GraphicalAppearance.getFromCsvValue(graphical_appearance_name.toString()).ordinal, 
        total = graphicalAppearanceEncodingLength,
    )
    
    articleEmbeddings[it.article_id] = articleTowerModel.predictSoftly(input).toList()
}

articleTowerModel.close()

articleEmbeddings

## 3. Loading data into database

The dataset doesn't contain values for brands, let's generate random values for these attribute type for now!

In [ ]:
import kotlin.random.Random

fun generateRandomBrandName(): String = when(Random.nextInt(3)) {
    0 -> "Cavin Klein"
    1 -> "Guccini"
    else -> "Pradell"
}

Let's insert the articles into the database first! To insert the data, Postgres JDBC driver is used!

In [ ]:
import kotlinx.datetime.Clock
import kotlinx.datetime.TimeZone
import kotlinx.datetime.toInstant
import kotlinx.datetime.toJavaInstant
import kotlinx.datetime.toJavaLocalDate
import kotlinx.datetime.toLocalDateTime
import java.sql.Connection
import java.sql.Date
import java.sql.DriverManager
import java.sql.PreparedStatement
import java.sql.Timestamp

var connection: Connection? = null
var preparedStatement: PreparedStatement? = null

try {
    connection = DriverManager.getConnection("jdbc:postgresql://localhost:5432/postgres", "postgres", "password")

    preparedStatement = connection?.prepareStatement("DELETE FROM articles")

    preparedStatement?.execute()

    preparedStatement?.close()
    
    val priceScaling = 30000

    val insertArticleSql = "INSERT INTO articles (id, name, price, brand, color, shade, graphical_appearance, index, garment_group, description, " +
            "creation_date_time, recommendation_embedding) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)"
    
    val timeZone = TimeZone.currentSystemDefault()

    articlesDf.forEach {
        preparedStatement = connection?.prepareStatement(insertArticleSql)

        preparedStatement?.setString(1, this@forEach.article_id.toString())
        preparedStatement?.setString(2, this@forEach.prod_name)
        preparedStatement?.setInt(3, (transactionsDf.first { it.article_id == this@forEach.article_id }.price * priceScaling.toBigDecimal()).toInt())
        preparedStatement?.setString(4, generateRandomBrandName())
        preparedStatement?.setInt(5, PerceivedColourMasterName.getFromCsvValue(this@forEach.perceived_colour_master_name).ordinal)
        preparedStatement?.setInt(6, Shade.getFromCsvValue(this@forEach.perceived_colour_value_name).ordinal)
        preparedStatement?.setInt(7, GraphicalAppearance.getFromCsvValue(this@forEach.graphical_appearance_name).ordinal)
        preparedStatement?.setInt(8, IndexName.getFromCsvValue(this@forEach.index_name).ordinal)
        preparedStatement?.setInt(9, GarmentGroupName.getFromCsvValue(this@forEach.garment_group_name).ordinal)
        preparedStatement?.setString(10, this@forEach.detail_desc ?: "")
        preparedStatement?.setTimestamp(11, Timestamp.from(Clock.System.now().toLocalDateTime(timeZone).toInstant(timeZone).toJavaInstant()))
        preparedStatement?.setObject(12, articleEmbeddings[this@forEach.article_id].toString(), java.sql.Types.OTHER)

        preparedStatement?.executeUpdate()

        preparedStatement?.close()
    }
} catch(t: Throwable) {
    println("Exception while inserting users: ${t.message}")
} finally {
    try {
        preparedStatement?.close()
        connection?.close()
    } catch (t: Throwable) {
        println("Exception while closing connections: ${t.message}")
    }
}

And lastly let's upload the user's data using JDBC again! The uploaded password hash for every user is the hash for the following value: password123456789

In [ ]:
import kotlinx.datetime.Clock
import kotlinx.datetime.TimeZone
import kotlinx.datetime.toLocalDateTime
import kotlinx.datetime.toJavaLocalDate
import java.sql.Connection
import java.sql.Date
import java.sql.DriverManager
import java.sql.PreparedStatement

var connection: Connection? = null
var preparedStatement: PreparedStatement? = null

try {
    connection = DriverManager.getConnection("jdbc:postgresql://localhost:5432/postgres", "postgres", "password")

    preparedStatement = connection?.prepareStatement("DELETE FROM application_users")
    
    preparedStatement?.execute()
    
    preparedStatement?.close()
    
    val insertUserSql = "INSERT INTO application_users (id, username, email, password, first_name, last_name, date_of_birth, recommendation_index) " +
            "VALUES (?, ?, ?, ?, ?, ?, ?, ?)"

    var customerIndex = 0
    
    val timeZone = TimeZone.currentSystemDefault()
    
    customersDf.forEach {
        preparedStatement = connection?.prepareStatement(insertUserSql)
        
        preparedStatement?.setString(1, this@forEach.customer_id)
        preparedStatement?.setString(2, "TestUser${customerIndex}")
        preparedStatement?.setString(3, "testuser${customerIndex}@test.test")
        preparedStatement?.setString(4, "\$argon2id\$v=19\$m=9216,t=4,p=1$/71ytTxXydnRoF1AeQItiCvpcw7LZTOg5rqWHlzP7SE\$KXxdf+CDl4iZi6nq4GPq1fGHp7tT/86pMl5jxK/1DWI")
        preparedStatement?.setString(5, "Test")
        preparedStatement?.setString(6, "User")
        preparedStatement?.setDate(7, Date.valueOf(Clock.System.now().toLocalDateTime(timeZone).date.toJavaLocalDate()))
        preparedStatement?.setInt(8, customerIndex)
        
        preparedStatement?.executeUpdate()
        
        customerIndex += 1
        
        preparedStatement?.close()
    }
} catch(t: Throwable) {
    println("Exception while inserting users: ${t.message}")
} finally {
    try {
        preparedStatement?.close()
        connection?.close()
    } catch (t: Throwable) {
        println("Exception while closing connections: ${t.message}")   
    }
}